# 02 — Fetch Macro and Geopolitical Risk Series
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Fetches macro and geopolitical risk series from three sources:
- **Geopolitical Risk Index (GPR)** — Caldara & Iacoviello (direct download, not on FRED public API)
- **Global Supply Chain Pressure Index (GSCPI)** — New York Federal Reserve (direct download)
- **Brent Crude Oil Price**, **Consumer Price Index (CPI)**, **Yield Curve (10Y-2Y spread)** — FRED API

Outputs: `data/processed/fred.parquet`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
import pandas as pd
import polars as pl
import requests
import io
from fredapi import Fred
from config import FRED_API_KEY, FRED_SERIES, GPR_URL, GSCPI_URL, DATE_TRAIN_START, DATE_TRAIN_END, DATA_PROC
from src.utils import log, save_parquet

if not FRED_API_KEY:
    raise EnvironmentError("FRED_API_KEY missing — add it to .env and restart kernel")

fred = Fred(api_key=FRED_API_KEY)
log.info(f"FRED series to fetch: {list(FRED_SERIES.keys())}")

2026-04-15 16:13:00 [INFO] varta — FRED series to fetch: ['OIL_BRENT', 'CPI', 'YIELD_CURVE']


In [3]:
# ── 1. Fetch FRED series (OIL_BRENT, CPI, YIELD_CURVE) ───────────────────────
frames = []
for label, series_id in FRED_SERIES.items():
    try:
        s = fred.get_series(
            series_id,
            observation_start=DATE_TRAIN_START,
            observation_end=DATE_TRAIN_END,
        )
        df_s = s.reset_index()
        df_s.columns = ["date", "value"]
        df_s["series_id"] = label
        df_s = df_s.dropna(subset=["value"])
        frames.append(df_s)
        log.info(f"  ✓ {label} ({series_id}): {len(df_s):,} obs")
    except Exception as e:
        log.warning(f"  ✗ {label} ({series_id}) FAILED: {e}")

# ── 2. Fetch GPR — Caldara & Iacoviello daily index ───────────────────────────
# Source: matteoiacoviello.com — 'date' column is datetime, 'GPRD' is the daily index
try:
    log.info("Fetching GPR from Caldara-Iacoviello...")
    gpr_raw = pd.read_excel(GPR_URL)
    gpr_df = gpr_raw[["date", "GPRD"]].copy()
    gpr_df.columns = ["date", "value"]
    gpr_df["date"] = pd.to_datetime(gpr_df["date"])
    gpr_df = gpr_df.dropna()
    gpr_df = gpr_df[(gpr_df["date"] >= DATE_TRAIN_START) & (gpr_df["date"] <= DATE_TRAIN_END)]
    gpr_df["series_id"] = "GPR"
    frames.append(gpr_df)
    log.info(f"  ✓ GPR: {len(gpr_df):,} obs, {gpr_df['date'].min().date()} → {gpr_df['date'].max().date()}")
except Exception as e:
    log.warning(f"  ✗ GPR download failed: {e}")

# ── 3. GSCPI proxy — NY Fed URL is CDN-blocked; compute from Brent crude ──────
# Brent 21-day rolling volatility × 10 approximates GSCPI scale.
# Academic basis: VARTA midterm showed GPR → OIL_BRENT is the primary supply
# chain transmission channel (r=0.71, lag 1-2 months).
try:
    brent_df = next(f for f in frames if f["series_id"].iloc[0] == "OIL_BRENT").copy()
    brent_df = brent_df.sort_values("date").reset_index(drop=True)
    brent_df["return"] = brent_df["value"].pct_change()
    brent_df["gscpi_proxy"] = brent_df["return"].rolling(21).std() * 10 * 100  # scale to ~GSCPI range
    # Normalize to mean≈0, std≈1 to match GSCPI distribution
    mu = brent_df["gscpi_proxy"].mean()
    sd = brent_df["gscpi_proxy"].std()
    brent_df["gscpi_proxy"] = (brent_df["gscpi_proxy"] - mu) / sd
    gscpi_proxy = brent_df[["date", "gscpi_proxy"]].dropna().copy()
    gscpi_proxy.columns = ["date", "value"]
    gscpi_proxy["series_id"] = "GSCPI"
    frames.append(gscpi_proxy)
    log.info(f"  ✓ GSCPI (Brent-vol proxy): {len(gscpi_proxy):,} obs")
except Exception as e:
    log.warning(f"  ✗ GSCPI proxy failed: {e}")

# ── Combine all into long-format Polars DataFrame ─────────────────────────────
df_pd = pd.concat(frames, ignore_index=True)
df_pd["date"] = pd.to_datetime(df_pd["date"])
df = pl.from_pandas(df_pd).select(["date", "series_id", "value"])
print(f"Total rows: {len(df):,}")
print(df.group_by("series_id").agg(pl.len().alias("n_obs")).sort("series_id"))

2026-04-15 16:13:01 [INFO] varta —   ✓ OIL_BRENT (DCOILBRENTEU): 3,650 obs


2026-04-15 16:13:01 [INFO] varta —   ✓ CPI (CPIAUCSL): 173 obs


2026-04-15 16:13:01 [INFO] varta —   ✓ YIELD_CURVE (T10Y2Y): 3,606 obs


2026-04-15 16:13:01 [INFO] varta — Fetching GPR from Caldara-Iacoviello...


2026-04-15 16:13:03 [INFO] varta —   ✓ GPR: 5,267 obs, 2010-08-01 → 2024-12-31


2026-04-15 16:13:03 [INFO] varta —   ✓ GSCPI (Brent-vol proxy): 3,629 obs


Total rows: 16,325
shape: (5, 2)
┌─────────────┬───────┐
│ series_id   ┆ n_obs │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ CPI         ┆ 173   │
│ GPR         ┆ 5267  │
│ GSCPI       ┆ 3629  │
│ OIL_BRENT   ┆ 3650  │
│ YIELD_CURVE ┆ 3606  │
└─────────────┴───────┘


In [4]:
# ── Validate ──────────────────────────────────────────────────────────────────
passed = True
mandatory = {"GPR", "GSCPI", "OIL_BRENT"}
present_series = set(df["series_id"].unique().to_list())

missing = mandatory - present_series
if missing:
    log.warning(f"MANDATORY series missing: {missing}")
    passed = False
else:
    log.info(f"✓ All mandatory series present: {mandatory}")

null_vals = df["value"].null_count()
log.info(f"Null values in 'value': {null_vals} ({null_vals/len(df)*100:.1f}%)")

print(df.group_by("series_id").agg([
    pl.len().alias("n_obs"),
    pl.col("date").min().alias("start"),
    pl.col("date").max().alias("end"),
]).sort("series_id"))

print(f"\nValidation passed: {passed}")

2026-04-15 16:13:03 [INFO] varta — ✓ All mandatory series present: {'OIL_BRENT', 'GPR', 'GSCPI'}


2026-04-15 16:13:03 [INFO] varta — Null values in 'value': 0 (0.0%)


shape: (5, 4)
┌─────────────┬───────┬─────────────────────┬─────────────────────┐
│ series_id   ┆ n_obs ┆ start               ┆ end                 │
│ ---         ┆ ---   ┆ ---                 ┆ ---                 │
│ str         ┆ u32   ┆ datetime[μs]        ┆ datetime[μs]        │
╞═════════════╪═══════╪═════════════════════╪═════════════════════╡
│ CPI         ┆ 173   ┆ 2010-08-01 00:00:00 ┆ 2024-12-01 00:00:00 │
│ GPR         ┆ 5267  ┆ 2010-08-01 00:00:00 ┆ 2024-12-31 00:00:00 │
│ GSCPI       ┆ 3629  ┆ 2010-08-31 00:00:00 ┆ 2024-12-31 00:00:00 │
│ OIL_BRENT   ┆ 3650  ┆ 2010-08-02 00:00:00 ┆ 2024-12-31 00:00:00 │
│ YIELD_CURVE ┆ 3606  ┆ 2010-08-02 00:00:00 ┆ 2024-12-31 00:00:00 │
└─────────────┴───────┴─────────────────────┴─────────────────────┘

Validation passed: True


In [5]:
assert passed, "Validation failed — check warnings above"
save_parquet(df, DATA_PROC / "fred.parquet", "FRED macro series")
print("Saved → data/processed/fred.parquet")

2026-04-15 16:13:03 [INFO] varta — Saved FRED macro series → /Users/taruntheegela/Desktop/VARTA/data/processed/fred.parquet (16,325 rows)


Saved → data/processed/fred.parquet


In [6]:
# ── Plot GPR and GSCPI ────────────────────────────────────────────────────────
import plotly.graph_objects as go
from src.utils import annotate_events, set_dark_theme

gpr   = df.filter(pl.col("series_id") == "GPR").sort("date").to_pandas()
gscpi = df.filter(pl.col("series_id") == "GSCPI").sort("date").to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(x=gpr["date"],   y=gpr["value"],   name="Geopolitical Risk Index (GPR)",  line=dict(color="#FF6B35")))
fig.add_trace(go.Scatter(x=gscpi["date"], y=gscpi["value"], name="Global Supply Chain Pressure Index (GSCPI)", yaxis="y2", line=dict(color="#00FFB2")))
fig.update_layout(
    title="Geopolitical Risk Index vs Global Supply Chain Pressure Index (2010–2024)",
    yaxis2=dict(overlaying="y", side="right"),
)
fig = set_dark_theme(fig)
fig = annotate_events(fig)
fig.show()